In [1]:
from vivarium import Artifact, InteractiveContext
import pandas as pd, numpy as np, os

In [2]:
! pip list | grep vivarium

# [software verion + hash . date]

vivarium                                 4.0.3
vivarium_build_utils                     3.2.2
vivarium_cluster_tools                   3.1.6
vivarium-compat                          0.6.1
vivarium-dependencies                    1.1.0
vivarium_gates_mncnh                     34.1.dev8+g5eaecaf6f.d20260616 /mnt/share/homes/tylerdy/vivarium_gates_mncnh
vivarium-gbd-mapping                     6.0.1
vivarium_public_health                   5.0.2
vivarium-risk-distributions              3.1.1
vivarium_testing_utils                   0.5.5


In [3]:
! pip freeze | grep vivarium

vivarium==4.0.3
vivarium-compat==0.6.1
vivarium-dependencies==1.1.0
vivarium-gbd-mapping==6.0.1
vivarium-risk-distributions==3.1.1
vivarium_build_utils==3.2.2
vivarium_cluster_tools==3.1.6
-e git+https://github.com/ihmeuw/vivarium_gates_mncnh.git@e52a301dd37c0103365ae1ac044c9a1312cdb0e6#egg=vivarium_gates_mncnh
vivarium_public_health==5.0.2
vivarium_testing_utils==0.5.5


In [4]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from vivarium import InteractiveContext, Artifact

In [20]:
import vivarium_gates_mncnh
from vivarium.framework.configuration import build_model_specification
from pathlib import Path

from vivarium_gates_mncnh.constants.data_values import (
    SIMULATION_EVENT_NAMES,
    COLUMNS,
    PREGNANCY_OUTCOMES,
    PIPELINES,
)

path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
custom_model_specification = build_model_specification(path)
custom_model_specification.configuration.input_data.input_draw_number = 60

custom_model_specification.configuration.population.population_size = 20_000 * 3

included_components = custom_model_specification.components.vivarium_gates_mncnh.components
included_components

['Hemoglobin()',
 'AnemiaInterventionPropensity()',
 'AgelessPopulation("population.scaling_factor")',
 'Pregnancy()',
 'ResultsStratifier()',
 'ANCAttendance()',
 'Ultrasound()',
 'MaternalDisorder("maternal_obstructed_labor_and_uterine_rupture")',
 'MaternalDisorder("maternal_hemorrhage")',
 'MaternalDisorder("maternal_sepsis_and_other_maternal_infections")',
 'ResidualMaternalDisorders()',
 'AbortionMiscarriageEctopicPregnancy()',
 'MaternalDisordersBurden()',
 'LBWSGRisk()',
 "LBWSGRiskEffect('cause.all_causes.all_cause_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_sepsis_and_other_neonatal_infections.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_with_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_without_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma.cause_specific_mortality_risk')",
 "PretermBirth('neonatal_preterm_birth_with_r

In [6]:
artifact_path = custom_model_specification.configuration.input_data.artifact_path
art = Artifact(artifact_path)

artifact_path

'/mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/model37.0/ethiopia.hdf'

In [7]:
draw_num = custom_model_specification.configuration.input_data.input_draw_number
draw = 'draw_' + str(draw_num)
draw

'draw_60'

CHECK: puerperal sepsis hemoglobin effect size in the artifact does not vary by sex, age, or year.

Suite: artifact tests.

Type: precise assert.

In [11]:
sepsis_on_hemoglobin_shift = art.load("cause.maternal_sepsis_and_other_maternal_infections.hemoglobin_shift")[draw]
assert len(sepsis_on_hemoglobin_shift) == 2  # early and late postpartum
sepsis_on_hemoglobin_shift

postpartum_period
early_postpartum   -13.247920
late_postpartum     -3.042343
Name: draw_60, dtype: float64

In [ ]:
def check_hemoglobin_shift(sim: InteractiveContext):
    pop = sim.get_population([COLUMNS.MATERNAL_SEPSIS, COLUMNS.PREGNANCY_OUTCOME])
    # We model maternal sepsis on live and still births only
    live_births = pop.loc[
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.LIVE_BIRTH_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.STILL_BIRTH_OUTCOME)
    ]
    has_sepsis = live_births[COLUMNS.MATERNAL_SEPSIS]
    sepsis_idx = live_births.index[has_sepsis]
    no_sepsis_idx = live_births.index[~has_sepsis]
    if len(sepsis_idx) == 0:
        return
    # Get hemoglobin exposure values from the pipeline
    hgb = sim.get_population(PIPELINES.HEMOGLOBIN_EXPOSURE).loc[live_births.index]
    mean_hgb_sepsis = hgb.loc[sepsis_idx].mean()
    mean_hgb_no_sepsis = hgb.loc[no_sepsis_idx].mean()

    # is step name the next step to run or the last step which was run? i couldnt check the hemoglobin exposure pipline when
    # steps_remaining was 0
    return {'step': sim._clock.step_name, 'mean_hgb_sepsis': mean_hgb_sepsis, 'mean_hgb_no_sepsis': mean_hgb_no_sepsis}
    # Sepsis shift is negative, so sepsis group should have lower hemoglobin
    #observed_diff = mean_hgb_sepsis - mean_hgb_no_sepsis
    #display(f"{sim._clock.step_name}: ({mean_hgb_sepsis} - {mean_hgb_no_sepsis}) = {observed_diff}")

    # TODO check for age groups as well

In [ ]:
def run_all_steps(sim: InteractiveContext):
    step_shifts = []
    while sim.get_number_of_steps_remaining() > 0:
        step_shift = check_hemoglobin_shift(sim)
        if step_shift:
            step_shifts.append(step_shift)
        sim.step()
    return pd.DataFrame(step_shifts)

In [35]:
def run_to_step(sim: InteractiveContext, step_name: str):
    while sim._clock.step_name != step_name:
        sim.step()
    return sim

In [43]:
sim = InteractiveContext(custom_model_specification)
step_shifts = run_all_steps(sim)

2026-07-09 08:42:16.993 | INFO     | simulation_7-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/model37.0/ethiopia.hdf.
2026-07-09 08:42:16.994 | INFO     | simulation_7-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-07-09 08:42:16.995 | INFO     | simulation_7-artifact_manager:82 - Artifact additional filter terms are None.
2026-07-09 08:43:16.261 | WARNING  | simulation_7-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-07-09 08:43:16.262 | WARNING  | simulation_7-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-07-09 08:43:16.683 | WARNING  | simulation_7-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-07-09 08:43:16.684 | W

In [52]:
step_shifts["observed_diff"] = step_shifts["mean_hgb_sepsis"] - step_shifts["mean_hgb_no_sepsis"]
step_shifts

,step,mean_hgb_sepsis,mean_hgb_no_sepsis,observed_diff
0,residual_maternal_disorders,106.770011,120.506701,-13.736690
1,abortion_miscarriage_ectopic_pregnancy,106.770011,120.506701,-13.736690
2,mortality,106.770011,120.506701,-13.736690
3,early_neonatal_mortality,93.522091,120.506701,-26.984611
4,late_neonatal_mortality,103.727668,120.506701,-16.779033
5,postpartum_depression,106.770011,120.506701,-13.736690


In [ ]:
sim.get_population(PIPELINES.)

0        74 days 19:01:06.429133865
1       126 days 23:31:15.581246692
2       110 days 17:55:58.891360316
3       107 days 05:14:00.166605357
4       276 days 11:52:33.669714005
                    ...            
59995   113 days 11:59:44.377486765
59996   270 days 13:27:37.609342824
59997   159 days 12:56:30.341807350
59998   273 days 06:28:27.698396036
59999   283 days 05:57:06.420313120
Name: pregnancy_duration, Length: 60000, dtype: timedelta64[ns]